# Coding Assignment: Did the New Checkout Page Work?
**Data Science: Foundations to Applications (SCD-4XX)** · Weeks 6–7 · Statistics I & II

**Total: 100 points** · Work individually · Estimated time: 2–3 hours

---
## The scenario
**ShopSphere**, an online retailer, redesigned its checkout page. For four weeks (1–28 Sept 2026), visitors were **randomly assigned** to the current page (`control`) or the redesign (`treatment`). You are the data scientist asked to advise leadership.

- The redesign cost **\$40,000** to build. Leadership will roll it out only if the **true conversion-rate lift is at least 1.0 percentage point (pp)**.
- The **primary metric** (declared before the test) is **conversion rate**. Time on page is a *secondary* metric.

## The data: `shopsphere_ab_test.csv` (keep it in the same folder as this notebook)
| Column | Meaning |
|---|---|
| `user_id` | Unique visitor ID |
| `visit_date` | Date of visit |
| `group` | `control` or `treatment` |
| `device` | `mobile`, `desktop` or `tablet` |
| `customer_type` | `new` or `returning` |
| `time_on_page_sec` | Seconds spent on the checkout page |
| `converted` | 1 if the visitor purchased, else 0 |
| `order_value` | Order value in dollars (only for converters; missing otherwise) |

## Rules & grading
- Write your code **in the empty code cells**, and your explanations **in the answer cells**. Use α = **0.05** throughout.
- Every task should work on its own (re-compute what you need from `df`), so the notebook runs **top to bottom**. Before submitting: `Kernel → Restart Kernel and Run All Cells`, and make sure there are no errors.
- Marks are given for correct code **and** correct interpretation. A p-value without a sentence explaining what it means earns at most half the points.
- Use `np.random.seed(2026)` once (in the setup cell) so your simulations are reproducible.
- **Submit** the executed `.ipynb` file (outputs visible).

| Part | Topic | Points |
|---|---|---|
| 1 | Descriptive statistics | 15 |
| 2 | Probability & distributions | 20 |
| 3 | CLT & confidence intervals | 15 |
| 4 | Hypothesis tests | 20 |
| 5 | Business recommendation | 20 |
| 6 | Statistical pitfalls | 10 |

## Part 0 · Setup & first look  *(ungraded, but required)*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

%matplotlib inline
sns.set_theme(style="whitegrid")
np.random.seed(2026)

df = pd.read_csv("shopsphere_ab_test.csv", parse_dates=["visit_date"])
print(df.shape)
df.head()

In [ ]:
# Explore: dtypes, missing values per column, number of users per group.
# Why does `order_value` have missing values? (You'll explain this in Task 1.3.)

---
## Part 1 · Descriptive Statistics  *(15 pts)*

### Task 1.1 · Summary statistics  *(5 pts)*

For `order_value` (converters only), compute the **mean, median, standard deviation, variance, range and IQR**, and the **mode of `device`**. Put the first six in one tidy `pandas` Series called `summary`.

In [ ]:
ov = df["order_value"].dropna()

# YOUR CODE HERE

### Task 1.2 · Visualize & find outliers  *(5 pts)*

Draw a **histogram** (with vertical lines for mean and median) and a **boxplot** of `order_value` side by side. Use the **1.5 × IQR rule** to count how many orders are outliers.

In [ ]:
# YOUR CODE HERE

### Task 1.3 · Interpret  *(5 pts)*

In 3–4 sentences: (a) Describe the **shape** of the distribution. (b) Which measure of centre and which measure of spread would you report to leadership, and why? (c) Why does `order_value` have missing values?

**Your answer:** *(double-click this cell and write here)*

---
## Part 2 · Probability & Distributions  *(20 pts)*

### Task 2.1 · Binomial  *(5 pts)*

Let `p_control` be the observed conversion rate in the **control** group. Suppose **200 new visitors** see the control page. Compute the probability that **at least 20** of them convert, (a) **exactly** with `scipy.stats.binom` and (b) by **simulation** with `np.random.binomial` (100,000 repetitions). Do they agree?

In [ ]:
# YOUR CODE HERE

### Task 2.2 · Poisson  *(5 pts)*

The support team receives on average **3.2 tickets per hour**. Compute (a) the probability of **no tickets** in an hour, (b) the probability of **more than 6 tickets** in an hour, (c) the **smallest staffing level `k`** such that the chance of *more than k* tickets in an hour is **at most 1%** (hint: `isf`). Check (b) with a simulation.

In [ ]:
# YOUR CODE HERE

### Task 2.3 · Conditional probability from data  *(5 pts)*

Using the full dataset, compute: `P(convert)`, `P(convert | returning)`, `P(convert | new)`, `P(mobile AND convert)`, and `P(convert | mobile)`. Then, in a comment or print statement, state whether **customer type and conversion look independent**.

In [ ]:
# YOUR CODE HERE

### Task 2.4 · Bayes' theorem: fraud detection  *(5 pts)*

2% of orders are fraudulent. A fraud model flags **90%** of fraudulent orders but also flags **5%** of legitimate ones. Write a function `p_fraud_given_flag(prevalence, sensitivity, false_positive_rate)` and use it to find the probability that a **flagged** order is really fraud. Then recompute for a prevalence of **0.2%**.

In [ ]:
def p_fraud_given_flag(prevalence, sensitivity, false_positive_rate):
    # YOUR CODE HERE
    pass

### Task 2.4b · Interpret  *(0 pts)*

*(Included in the 5 points of Task 2.4.)* In 2 sentences: why is a flagged order still more likely to be legitimate than fraudulent, and what does this mean for how the company should use the flag?

**Your answer:** *(double-click this cell and write here)*

---
## Part 3 · Central Limit Theorem & Confidence Intervals  *(15 pts)*

### Task 3.1 · CLT in action  *(5 pts)*

Treat the observed `order_value` values as a skewed population. For sample sizes **n = 5, 30 and 100**, draw **1,000 samples** (with `np.random.choice`), record each sample's mean, and plot the three histograms side by side. Print the **skewness** (`stats.skew`) and the **standard deviation of the sample means** for each `n`, next to the theoretical `σ/√n`.

In [ ]:
# YOUR CODE HERE

### Task 3.1b · Interpret  *(0 pts)*

*(Included in Task 3.1.)* In 2–3 sentences, explain what the plots show and why this matters for confidence intervals on **mean order value**.

**Your answer:** *(double-click this cell and write here)*

### Task 3.2 · CI for mean order value  *(5 pts)*

Compute a **95% t-based confidence interval** for the mean `order_value` separately for the control and treatment groups (converters only). Print the sample size, mean and interval for each.

In [ ]:
# YOUR CODE HERE

### Task 3.3 · CI for the difference in conversion rates  *(5 pts)*

Write a function `diff_prop_ci(x1, n1, x2, n2, conf=0.95)` returning the CI for `p2 - p1` using the normal approximation `diff ± z × SE` with `SE = sqrt(p1(1-p1)/n1 + p2(1-p2)/n2)` (get `z` from `stats.norm.ppf`). Apply it to control (group 1) vs treatment (group 2) at **95%** and **99%**. What happens to the width, and why?

In [ ]:
def diff_prop_ci(x1, n1, x2, n2, conf=0.95):
    # YOUR CODE HERE
    pass

---
## Part 4 · Hypothesis Testing  *(20 pts)*

### Task 4.1 · Hypotheses  *(3 pts)*

For the **primary metric**, write H₀ and H₁ in words *and* symbols (use `p_C` and `p_T` for the true conversion rates), say whether the test is one- or two-tailed, and justify it.

**Your answer:** *(double-click this cell and write here)*

### Task 4.2 · Primary metric test  *(5 pts)*

Build the `group × converted` cross-tab and run the appropriate test. Report the test statistic, the p-value, and your decision. (Run it with the default continuity correction **and** with `correction=False`; do they lead to the same decision?)

In [ ]:
# YOUR CODE HERE

### Task 4.3 · Secondary metric: time on page  *(4 pts)*

Run a **Welch t-test** (`equal_var=False`) on `time_on_page_sec`, control vs treatment. Also report the mean difference in seconds and **Cohen's d** (`d = mean difference / sqrt((s₁² + s₂²)/2)`). Is the result statistically significant? Is it practically significant? Explain in a markdown cell below your code.

In [ ]:
# YOUR CODE HERE

### Task 4.3b · Statistical vs practical significance  *(0 pts)*

*(Part of Task 4.3.)* Interpret the result above.

**Your answer:** *(double-click this cell and write here)*

### Task 4.4 · ANOVA across devices  *(5 pts)*

Test whether **mean time on page differs by device** (3 groups). Use the right test, then a **Tukey HSD** post-hoc test. Which pairs of devices differ? (Remember: groups from `df.groupby('device')` are ordered alphabetically: desktop, mobile, tablet.) Why not run three t-tests instead?

In [ ]:
# YOUR CODE HERE

### Task 4.5 · Did randomization work?  *(3 pts)*

A valid A/B test needs comparable groups. Run chi-square tests of **`group` vs `device`** and **`group` vs `customer_type`**. What result do you *hope* to see, and what do you conclude?

In [ ]:
# YOUR CODE HERE

---
## Part 5 · From Statistics to a Business Recommendation  *(20 pts)*

### Task 5.1 · Effect size vs the business threshold  *(5 pts)*

Compute the **absolute lift** (in pp), the **relative lift** (%), and the **95% CI** for the lift (reuse your `diff_prop_ci` or recompute). Then answer in code: (i) does the CI exclude 0? (ii) does the CI **lower bound** reach the 1.0 pp business threshold? (iii) does the CI **upper bound** reach it?

In [ ]:
THRESHOLD_PP = 1.0

# YOUR CODE HERE

### Task 5.2 · Power analysis  *(5 pts)*

Using `statsmodels` (`proportion_effectsize`, `NormalIndPower`): (a) how many users **per group** would be needed to detect a **1.0 pp lift** over the control baseline with **80% power** at α = 0.05? (b) What was the **achieved power** of this experiment (5,000 users per group) for a true 1.0 pp lift? (c) The site sends ≈ 1,250 users per group per week: how many weeks would the properly powered test need?

In [ ]:
# YOUR CODE HERE

### Task 5.3 · The leadership memo  *(10 pts)*

Write a **memo of at most 200 words** to the VP of Product. It must: (1) **open with a clear recommendation** (ship / don't ship / keep testing) in plain English; (2) report the lift with its **95% CI** and say what the p-value means in one sentence (no jargon); (3) address **practical significance** against the 1.0 pp rule and the secondary metric (time on page); (4) state **what you would do next** and one limitation or risk. *Grading:* correctness of the statistics (4), appropriate recommendation given the evidence (3), clarity for a non-technical reader (2), within word limit (1).

**Your answer:** *(double-click this cell and write here)*

---
## Part 6 · Statistical Pitfalls  *(10 pts)*

### Task 6.1 · The danger of peeking  *(5 pts)*

Simulate an **A/A test**: both groups have the *same* true conversion rate (9%), so any 'significant' result is a **false positive**. In each of 2,000 simulated experiments with 5,000 users per group, run a chi-square test (`correction=False`) at **5 interim looks** (n = 1,000, 2,000, 3,000, 4,000, 5,000 per group). Compare: (a) the false-positive rate if you **only analyze at the final look**, versus (b) if you **stop and declare victory the first time p < 0.05** at any look. Complete the function below; the expected answer for (a) is about 5%.

In [ ]:
def simulate_peeking(n_sims=2000, p=0.09, looks=(1000, 2000, 3000, 4000, 5000), alpha=0.05):
    # For each simulation: draw 5,000 Bernoulli(p) outcomes for group A and for group B,
    # compute the p-value at each look using the first n users of each group,
    # and record (any look significant?, final look significant?).
    # Return (peeking_false_positive_rate, final_only_false_positive_rate)
    pass

# peek_rate, final_rate = simulate_peeking()
# print(...)

### Task 6.2 · Multiple comparisons  *(5 pts)*

A team tracks **20 metrics** and declares a win if *any* metric has p < 0.05. Simulate this when there is **no real effect on any metric** (compare two samples of 200 draws from `np.random.normal` for each metric; 2,000 simulated experiments). Report (a) the simulated probability of at least one false positive, (b) the theoretical value `1 − 0.95²⁰`, and (c) the rate when using the **Bonferroni** threshold `α/20`.

In [ ]:
# YOUR CODE HERE

### Task 6.3 · Reflection  *(0 pts)*

*(Included in Tasks 6.1–6.2.)* In 3–4 sentences, explain how these two pitfalls could have affected **this** A/B test if ShopSphere had checked results daily and also reported eight secondary metrics.

**Your answer:** *(double-click this cell and write here)*

---
## Submission checklist
- [ ] `Kernel → Restart Kernel and Run All Cells` completes **without errors**
- [ ] Every answer cell is filled in (interpretation, not just code)
- [ ] Memo (Task 5.3) is **≤ 200 words**
- [ ] File saved as `Assignment_<YourStudentID>.ipynb` with outputs visible